# webAI-ColVec1.1 — visual arm for ViDoRe V3 physics — **Kaggle T4 x2**

Port of `webAI_ColVec_visual_arm_physics.ipynb` (Colab) to Kaggle. Same output:
a 302 x 1674 MaxSim score matrix (`physics_colvec_export.zip`) that
`research/experiments/physics_kdl_slate.py --parse pdf-inspector --visual-name colvec`
scores locally.

**Why re-run at all** (the visual arm is parse-independent, and the 8b run we have
already gave SEP+ColVec = 53.15): the one lever left on the visual side is
`MAX_VISUAL_TOKENS`. At 144 DPI / 1792 tokens the processor already downscales
every page to its 1.835 Mpx budget — more DPI is pixel-identical. Raising the
token budget (3584 -> 3.67 Mpx, ~220 DPI equiv) is the actual "let the model see
the equations" test. Set `MAX_VISUAL_TOKENS = 1792` to just reproduce; `3584` for
the experiment. **Ledger §25.**

**⚠️ LICENSE webAI Non-Commercial v1.0** — research only. Commercial swap:
`nvidia/nemotron-colembed-vl-4b-v2`.

---

### Kaggle setup (differs from Colab — read this)

1. **Data as a Kaggle Dataset.** Locally:
   ```bash
   cd AXIOM_DE-RD
   git archive --format=zip -o ~/Desktop/colvec_bundle.zip HEAD
   zip -r ~/Desktop/colvec_bundle.zip \
     data/raw/benchmarks/vidore_v3_physics \
     data/benchmark/vidore_v3/physics \
     data/benchmark/vidore_v3/results/physics_KDL_pool.json
   ```
   Kaggle → **Datasets → New Dataset** → upload `colvec_bundle.zip` (it auto-unzips).
   Note the slug, e.g. `khoatran/colvec-bundle`.
2. **Notebook → Add Input →** your dataset. It mounts read-only at
   `/kaggle/input/<slug>/`.
3. **Settings → Accelerator → GPU T4 x2.** (P100 is one 16GB card — 8b will not fit
   without 8-bit; T4 x2 = 32GB and shards cleanly.)
4. **Settings → Internet → On** (needs a phone-verified account). Required for
   `pip install` and the HuggingFace model download. If you can't enable it,
   add `webAI-Official/webAI-ColVec1.1-8b` as a **model input** instead and set
   `MODEL_ID` to its mounted path.
5. **Test run first:** `TEST_RUN = True`, "Run All" interactively (~10 min). It
   checks the device map, dtype sanity, MaxSim spread, and prints a full-run time
   estimate against Kaggle's ~11 h usable session.
6. **Overnight:** set `TEST_RUN = False`, then **Save Version → Save & Run All
   (Commit)** — headless, survives the browser closing. Watch the **weekly** GPU
   quota (~30 h), not just the 12 h/session cap.
7. **Resume after a 12 h timeout:** Add Input → your own previous notebook version
   (its `/kaggle/working` becomes `/kaggle/input/<notebook-slug>/`); the checkpoint
   cell picks up `colvec_page_embeddings.pkl` from there.

In [ ]:
!nvidia-smi

## 1. Mount the bundle, copy the repo to a writable path

In [ ]:
import shutil, sys
from pathlib import Path

# find the dataset dir that contains the repo (has pyproject.toml)
SRC = None
for c in sorted(Path("/kaggle/input").glob("*")):
    if (c / "pyproject.toml").is_file():
        SRC = c; break
    for sub in c.glob("*"):
        if (sub / "pyproject.toml").is_file():
            SRC = sub; break
assert SRC, "repo not found under /kaggle/input -- did you Add Input the colvec_bundle dataset?"
print("bundle:", SRC)

REPO = Path("/kaggle/working/AXIOM_DE-RD")
if not REPO.exists():
    shutil.copytree(SRC, REPO)          # /kaggle/input is read-only; pip install -e needs write
print("repo ->", REPO)

pdfs = list((REPO / "data/raw/benchmarks/vidore_v3_physics").glob("*.pdf"))
assert len(pdfs) == 42, f"expected 42 PDFs, found {len(pdfs)}"
assert (REPO / "pyproject.toml").is_file()
POOL = REPO / "data/benchmark/vidore_v3/results/physics_KDL_pool.json"
print(f"OK: {len(pdfs)} PDFs, pool={'present' if POOL.is_file() else 'MISSING (cell 21 will skip)'}")

In [ ]:
%cd /kaggle/working/AXIOM_DE-RD
%pip install -q -e "." pymupdf
# webAI-ColVec ships trust_remote_code modules; config.json pins transformers 5.14.1
# (model_type "qwen3_5"), from the repo's evaluation-requirements-cu128.txt:
%pip install -q -U "transformers==5.14.1" "sentence-transformers==5.6.0" "accelerate==1.14.0"
%pip install -q -U bitsandbytes
print("deps installed. If pip reports a conflict, Run > Restart & Clear Cell Outputs, "
      "then run from the next cell (/kaggle/working survives a restart).")

## 2. Config

| knob | reproduce §25 (53.15) | resolution experiment |
|---|---|---|
| `MODEL_ID` | `...-8b` | `...-8b` |
| `MAX_VISUAL_TOKENS` | 1792 | 3584 |
| `DPI` | 144 | 224 |

`8b` fp16 weights ≈ 16–18 GB → needs **T4 x2** with `device_map="auto"`, or
`LOAD_8BIT=True` on one card, or drop to `-4b` (~9 GB, fits one T4).

In [ ]:
import torch

MODEL_ID          = "webAI-Official/webAI-ColVec1.1-8b"   # or ".../webAI-ColVec1.1-4b"
MAX_VISUAL_TOKENS = 1792     # 1792 = reproduce; 3584 = give the model finer patches
DPI               = 144      # 144 saturates 1792 tokens; raise with the token budget (224 for 3584)
LOAD_8BIT         = False    # True -> fits one T4, ~1.5x slower, small quality hit
TEST_RUN          = True     # True -> 24 pages / 8 queries, no export; flip for the overnight commit

n_gpu = torch.cuda.device_count()
props = [torch.cuda.get_device_properties(i) for i in range(n_gpu)]
for i, p in enumerate(props):
    print(f"GPU{i}: {p.name}  {p.total_memory/1024**3:.1f} GiB")
DTYPE = torch.float16       # T4 has no bf16
per_gpu = min(p.total_memory for p in props) / 1024**3
MAX_MEM = {i: f"{per_gpu-2:.0f}GiB" for i in range(n_gpu)} | {"cpu": "8GiB"}
print(f"MODEL_ID={MODEL_ID}  tokens={MAX_VISUAL_TOKENS}  DPI={DPI}  8bit={LOAD_8BIT}  test={TEST_RUN}")
print(f"device_map budget: {MAX_MEM}")
if "8b" in MODEL_ID and n_gpu == 1 and not LOAD_8BIT:
    print("!! 8b on a single GPU: set LOAD_8BIT=True or MODEL_ID=...-4b")

## 3. Render page images

In [ ]:
%cd /kaggle/working/AXIOM_DE-RD
!python research/experiments/render_page_images.py --dpi {DPI}
from pathlib import Path
images = sorted(Path("data/work/vidore_physics_page_images").glob("*.png"))
print(f"{len(images)} page images @ {DPI} DPI")
assert len(images) == 1674, f"expected 1674, got {len(images)}"

## 4. Load webAI-ColVec

`device_map="auto"` shards across both T4s. Verified against the repo's real
`modeling_colqwen35_bidirection.py`: `model(**inp)` returns a bare L2-normed
tensor (the `embed()` helper also covers ModelOutput/tuple). `score_retrieval`
exists but we compute MaxSim by hand in cell 8 (device-agnostic).

In [ ]:
from transformers import AutoModel, AutoProcessor

_kw = dict(trust_remote_code=True, attn_implementation="sdpa",
           device_map="auto", max_memory=MAX_MEM)
if LOAD_8BIT:
    from transformers import BitsAndBytesConfig
    _kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    _kw["dtype"] = DTYPE

processor = AutoProcessor.from_pretrained(
    MODEL_ID, trust_remote_code=True, max_num_visual_tokens=MAX_VISUAL_TOKENS)
model = AutoModel.from_pretrained(MODEL_ID, **_kw).eval()

dm = getattr(model, "hf_device_map", {})
print("hf_device_map:", dm)
assert not any(str(v) in ("cpu", "disk") for v in dm.values()), \
    "part of the model is on CPU/disk -> will be ~100x slower. Use LOAD_8BIT or -4b."
print("params:", sum(p.numel() for p in model.parameters()) / 1e9, "B")
IN_DEV = "cuda:0"

def embed(inp):
    out = model(**inp)
    if hasattr(out, "embeddings"): out = out.embeddings
    elif hasattr(out, "last_hidden_state"): out = out.last_hidden_state
    elif isinstance(out, (tuple, list)): out = out[0]
    return out

## 5. Test run — device map, dtype sanity, timing estimate

In [ ]:
import time, numpy as np
from PIL import Image

probe_imgs = images[:24]
t0 = time.time()
pe = []
for p in probe_imgs:
    inp = processor.process_images([Image.open(p).convert("RGB")]).to(IN_DEV)
    with torch.inference_mode():
        e = embed(inp)[0]
    pe.append(e.float().cpu())
per_img = (time.time() - t0) / len(probe_imgs)

# visual tokens actually fed (post-resize): seq_len of the page embedding
tok = [t.shape[0] for t in pe]
print(f"visual tokens/page: min={min(tok)} median={int(np.median(tok))} max={max(tok)}  (budget {MAX_VISUAL_TOKENS})")
print(f"embed dim: {pe[0].shape[-1]}   finite: {all(torch.isfinite(t).all().item() for t in pe)}")
print(f"{per_img:.2f}s/page  ->  1674 pages ~= {per_img*1674/60:.0f} min")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/AXIOM_DE-RD")
from src.evaluation.benchmarks import load

bench = load("vidore_v3", subset="physics", language="french")
qrels = bench.qrels()
questions = [q for q in bench.questions() if qrels.get(q.qid)]
print(f"{len(questions)} queries with qrels")

qi = processor.process_queries([q.query for q in questions[:8]]).to(IN_DEV)
with torch.inference_mode():
    qe = embed(qi)
qe0 = qe[0].float().cpu()
d0 = pe[0]
sim = (qe0 @ d0.T).max(dim=1).values.sum().item()
sim_far = (qe0 @ pe[-1].T).max(dim=1).values.sum().item()
print(f"MaxSim q0.d0={sim:.3f}  q0.d_last={sim_far:.3f}  (should differ -> scores discriminate)")
print(f"query finite: {torch.isfinite(qe).all().item()}   q tokens: {qe0.shape[0]}")

USABLE_MIN = 11 * 60
est = per_img * (1674 + len(questions)) / 60
print(f"\nFULL RUN estimate: ~{est:.0f} min  vs  ~{USABLE_MIN} min usable/session")
print("OK for one session" if est < USABLE_MIN * 0.85 else
      "!! exceeds one session -> the checkpoint resumes; add the prior version as Input and re-commit")
assert not TEST_RUN, "TEST_RUN=True: stop here, review the numbers above, then set TEST_RUN=False and commit."

## 6. Encode all pages — checkpoint every 100, resumable

In [ ]:
import pickle
CKPT = Path("/kaggle/working/colvec_page_embeddings.pkl")
RESUME = sorted(Path("/kaggle/input").glob("*/colvec_page_embeddings.pkl"))
if not CKPT.exists() and RESUME:
    shutil.copy(RESUME[-1], CKPT); print("resuming from", RESUME[-1])
page_emb = pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}
print(f"have {len(page_emb)}/{len(images)}")

for i, p in enumerate(images):
    key = p.stem.replace("__", "::")
    if key in page_emb:
        continue
    inp = processor.process_images([Image.open(p).convert("RGB")]).to(IN_DEV)
    with torch.inference_mode():
        page_emb[key] = embed(inp)[0].to(torch.float16).cpu()
    if (i + 1) % 100 == 0:
        pickle.dump(page_emb, open(CKPT, "wb")); print(f"  {i+1}/{len(images)}", flush=True)
pickle.dump(page_emb, open(CKPT, "wb"))
print(f"done: {len(page_emb)} page embeddings")

In [ ]:
query_emb = {}
for s in range(0, len(questions), 8):
    chunk = questions[s:s+8]
    inp = processor.process_queries([q.query for q in chunk]).to(IN_DEV)
    with torch.inference_mode():
        e = embed(inp)
    for q, v in zip(chunk, e):
        query_emb[q.qid] = v.to(torch.float16).cpu()
print(f"{len(query_emb)} query embeddings")

## 7. MaxSim score matrix (manual, on cuda:0)

In [ ]:
keys = sorted(page_emb)
qids = [q.qid for q in questions]
Q = torch.nn.utils.rnn.pad_sequence([query_emb[q] for q in qids], batch_first=True).to("cuda:0", torch.float16)
qmask = torch.nn.utils.rnn.pad_sequence(
    [torch.ones(query_emb[q].shape[0]) for q in qids], batch_first=True).to("cuda:0")
score_matrix = np.zeros((len(qids), len(keys)), dtype=np.float32)

for c in range(0, len(keys), 64):
    for j, k in enumerate(keys[c:c+64]):
        d = page_emb[k].to("cuda:0", torch.float16)
        sim = torch.einsum("nld,md->nlm", Q, d)
        score_matrix[:, c+j] = ((sim.max(dim=2).values * qmask).sum(dim=1)).float().cpu().numpy()
    if (c // 64) % 5 == 0:
        print(f"  {c+64}/{len(keys)}", flush=True)
print("score matrix:", score_matrix.shape)

try:
    ref = processor.score_retrieval([query_emb[qids[0]].float()], [page_emb[k].float() for k in keys[:50]])
    ref = np.asarray(ref.cpu() if hasattr(ref, "cpu") else ref).ravel()
    print("sanity vs processor.score_retrieval, corr:", np.corrcoef(ref, score_matrix[0, :50])[0, 1])
except Exception as e:
    print("(manual MaxSim only):", type(e).__name__)

## 8. Quick in-notebook check

In [ ]:
import math
def ndcg10(ranked, gold):
    dcg = sum((2**gold.get(k,0)-1)/math.log2(i+2) for i,k in enumerate(ranked[:10]))
    idcg = sum((2**g-1)/math.log2(i+2) for i,g in enumerate(sorted(gold.values(), reverse=True)[:10]))
    return 100*dcg/idcg if idcg>0 else 0.0

kx = {k:i for i,k in enumerate(keys)}
vals = [ndcg10(sorted(keys, key=lambda k:-score_matrix[r,kx[k]]), qrels[q]) for r,q in enumerate(qids)]
print(f"ColVec visual-only NDCG@10 = {sum(vals)/len(vals):.2f}   (n={len(vals)})")
print("ref: text pdf-inspector α=0.7 = 43.45 | ColVec-8b@1792 visual-only = 51.55 | SEP+ColVec = 53.15")

In [ ]:
import json as _json
if POOL.is_file():
    pool = _json.loads(POOL.read_text())["queries"]
    found = total = 0
    for r, q in enumerate(qids):
        if q not in pool: continue
        deep = {k for k,v in qrels[q].items() if v>0} - set(pool[q]["candidates"][:100])
        if not deep: continue
        top100 = set(sorted(keys, key=lambda k:-score_matrix[r,kx[k]])[:100])
        total += len(deep); found += len(deep & top100)
    print(f"gold the text pool misses (rank>=100): {total};  recovered in ColVec top-100: {found} "
          f"({100*found/max(total,1):.1f}%)")
else:
    print("no pool bundled -> skipped")

## 9. Export

In [ ]:
OUT = Path("/kaggle/working/physics_colvec_export"); OUT.mkdir(exist_ok=True)
np.save(OUT / "physics_colvec_scores.npy", score_matrix)
_json.dump(keys, open(OUT / "physics_colvec_keys.json", "w"))
_json.dump(qids, open(OUT / "physics_colvec_qids.json", "w"))
_json.dump({"model": MODEL_ID, "dpi": DPI, "max_visual_tokens": MAX_VISUAL_TOKENS,
            "load_8bit": LOAD_8BIT, "dim": int(next(iter(page_emb.values())).shape[-1]),
            "n_pages": len(keys), "n_queries": len(qids)}, open(OUT / "physics_colvec_meta.json", "w"))
shutil.make_archive("/kaggle/working/physics_colvec_export", "zip", OUT)
print("wrote /kaggle/working/physics_colvec_export.zip")
print("Get it: notebook Output tab (after commit), or  kaggle kernels output <user>/<kernel> -p .")

## Next (local, no GPU)

```bash
unzip physics_colvec_export.zip -d data/work/vidore_physics_colvec/
python research/experiments/physics_kdl_slate.py --parse pdf-inspector \
  --visual-dir data/work/vidore_physics_colvec --visual-name colvec --wv 0.8
```

Compare against §25: baseline 43.45 · SEP 45.52 · **SEP+ColVec@1792 = 53.15** ·
Nemotron 47.42. If `MAX_VISUAL_TOKENS=3584` beats 53.15 by more than ~0.7 and
the visual-only row climbs, the resolution experiment paid off — then also check
whether a reranker still can't sit on top (§25 redundancy).